# M0 smoke test — Qwen3-0.6B

This notebook is a thin runner for `src/m0_smoke_test.py`. Connect it to a Kaggle GPU kernel. It performs inference only: no training, weight modification, merging, or artifact upload.

In [1]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DevulapalliTharun/persistent-llm-unlearning.git"
REPO_DIR = Path("/kaggle/working/persistent-llm-unlearning")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Cloning into '/kaggle/working/persistent-llm-unlearning'...


Repository commit: d2e9937d1222c0651ba0ecbf8a42096a9f059ae9


In [2]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements/m0-smoke.txt"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m0-smoke.txt'], returncode=0)

In [3]:
import subprocess
import sys

OUTPUT_DIR = "/kaggle/working/m0-smoke"
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.m0_smoke_test",
        "--output-dir",
        OUTPUT_DIR,
    ],
    check=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 787.58it/s]


{
  "finished_at_utc": "2026-11-04T15:48:47.278294+00:00",
  "git_commit": "d2e9937d1222c0651ba0ecbf8a42096a9f059ae9",
  "inference": {
    "decoding": {
      "do_sample": true,
      "temperature": 0.7,
      "top_k": 20,
      "top_p": 0.8
    },
    "generated_tokens": 6,
    "generation_seconds": 2.548,
    "input_tokens": 24,
    "load_seconds": 15.1902,
    "max_new_tokens": 64,
    "prompt": "In one short sentence, state that the model loaded successfully.",
    "response": "The model loaded successfully.",
    "seed": 42
  },
  "model": {
    "class": "Qwen3ForCausalLM",
    "dtype": "torch.bfloat16",
    "id": "Qwen/Qwen3-0.6B",
    "parameter_count": 596049920,
    "requested_revision": "main",
    "resolved_revision": "c1899de289a04d12100db370d81485cdf75e47ca"
  },
  "runtime": {
    "accelerate": "1.15.0",
    "cuda_available": true,
    "cuda_runtime": "13.0",
    "gpu_count": 2,
    "gpus": [
      "Tesla T4",
      "Tesla T4"
    ],
    "huggingface_hub": "1.33.0",
    

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.m0_smoke_test', '--output-dir', '/kaggle/working/m0-smoke'], returncode=0)

In [4]:
import json
from pathlib import Path

result = json.loads(Path(OUTPUT_DIR, "result.json").read_text())
assert result["success"] is True
assert result["runtime"]["cuda_available"] is True
assert result["model"]["resolved_revision"]
result

{'finished_at_utc': '2026-11-04T15:48:47.278294+00:00',
 'git_commit': 'd2e9937d1222c0651ba0ecbf8a42096a9f059ae9',
 'inference': {'decoding': {'do_sample': True,
   'temperature': 0.7,
   'top_k': 20,
   'top_p': 0.8},
  'generated_tokens': 6,
  'generation_seconds': 2.548,
  'input_tokens': 24,
  'load_seconds': 15.1902,
  'max_new_tokens': 64,
  'prompt': 'In one short sentence, state that the model loaded successfully.',
  'response': 'The model loaded successfully.',
  'seed': 42},
 'model': {'class': 'Qwen3ForCausalLM',
  'dtype': 'torch.bfloat16',
  'id': 'Qwen/Qwen3-0.6B',
  'parameter_count': 596049920,
  'requested_revision': 'main',
  'resolved_revision': 'c1899de289a04d12100db370d81485cdf75e47ca'},
 'runtime': {'accelerate': '1.15.0',
  'cuda_available': True,
  'cuda_runtime': '13.0',
  'gpu_count': 2,
  'gpus': ['Tesla T4', 'Tesla T4'],
  'huggingface_hub': '1.33.0',
  'peak_gpu_memory_gib': 1.311,
  'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39',
  'python': '3.13.16'

In [5]:
from importlib.metadata import version

for package in ["accelerate", "huggingface-hub", "safetensors", "transformers"]:
    print(package, version(package))

accelerate 1.15.0
huggingface-hub 1.33.0
safetensors 0.8.0
transformers 5.20.2


## Completion rule

M0 is not complete merely because these cells run once. Review `result.json`, copy the small result files into a dated experiment record in Git, pin the verified dependency versions, and repeat the runner as a committed Kaggle version.